# MVP - Data Engineering

## Ingestão de Dados — CSV para Volume

Este notebook copia os arquivos CSV do repositório Git (pasta `Files/`) para o volume do Unity Catalog (`mvp_pucrio.raw_files.csv_files`).

**Fluxo:** GitHub → Git folder (pull) → cópia para volume UC → Bronze (carga para Delta)

> ⚠️ Antes de executar, certifique-se de que o Git folder está atualizado com `git pull`.

In [0]:
%sql
-- Garantir que o catálogo, o schema e o volume existem antes de copiar os arquivos
CREATE CATALOG IF NOT EXISTS mvp_pucrio;

CREATE SCHEMA IF NOT EXISTS mvp_pucrio.raw_files;

CREATE VOLUME IF NOT EXISTS mvp_pucrio.raw_files.csv_files;

In [0]:
import os

# Caminhos
git_folder = "/Workspace/Users/cyntia_invernizzi@hotmail.com/_DataEng_PUCRIO/Files"
volume_path = "/Volumes/mvp_pucrio/raw_files/csv_files"

# Listar CSVs no Git folder
csv_files = [f for f in os.listdir(git_folder) if f.endswith(".csv")]
print(f"Encontrados {len(csv_files)} arquivos CSV no Git folder:")
for f in sorted(csv_files):
    print(f"  - {f}")

In [0]:
# Copiar cada arquivo para o volume
import shutil

copied = 0
for filename in sorted(csv_files):
    src = os.path.join(git_folder, filename)
    dst = os.path.join(volume_path, filename)
    shutil.copy2(src, dst)
    copied += 1
    print(f"✓ {filename} copiado")

print(f"\n{copied} arquivos copiados para {volume_path}")

In [0]:
# Verificar arquivos no volume
volume_files = [f for f in os.listdir(volume_path) if f.endswith(".csv")]
print(f"Arquivos no volume ({len(volume_files)}):")
for f in sorted(volume_files):
    size_mb = os.path.getsize(os.path.join(volume_path, f)) / (1024 * 1024)
    print(f"  - {f} ({size_mb:.1f} MB)")

# Validar que todos os CSVs do Git estão no volume
missing = set(csv_files) - set(volume_files)
if missing:
    print(f"\n⚠️ Arquivos faltando: {missing}")
else:
    print("\n✅ Todos os CSVs do Git folder estão no volume.")